In [2]:
"""
FASE 3: Reprocessamento da simulação no OpenSim (ID, SO e JRA) e novo dataset
==============================================================================
Dissertação: LCA / cutting / OpenSim / SVM linear (PPGMSB, UNEB)

POR QUE ESTE SCRIPT EXISTE
  O diagnóstico de 11/11/2026 mostrou que a simulação anterior gerou momentos e
  forças de 15 a 65 vezes maiores que os da Dinâmica Inversa publicada pelos
  autores da base (Nitschke et al.). As causas encontradas foram:
    1. setup_forcas_vcut.xml com os PÉS TROCADOS: a força do pé direito
       (colunas ground_force_*) era aplicada no calcâneo esquerdo e vice-versa.
       Isso cria braços de alavanca enormes e infla momentos e reações.
    2. Templates presos ao Participante 02, trial 0122 (modelo e janela de tempo
       fixos); o batch só trocava caminhos.
    3. Ausência de Otimização Estática: o JRA rodava sem forças musculares.
    4. Mistura de fontes: JRA com coordenadas de coordinate tracking; ângulos de
       inverse methods; momento do quadril da Dinâmica Inversa dos autores.
    5. Pico extraído em toda a janela, inclusive na fase aérea.

O QUE ESTE SCRIPT FAZ, PARA CADA TRIAL DE V-CUT
  1. Usa o modelo escalonado do próprio participante (Participant_XX.osim).
  2. Usa as coordenadas de inverse methods dos autores e as GRF MEDIDAS,
     filtradas por Butterworth de 3a ordem, passagem dupla, 15 Hz (como os autores).
  3. Monta as cargas externas com o mapeamento CORRETO:
       ground_force_*   -> calcn_r (pé direito; contato inicial do movimento)
       l_ground_force_* -> calcn_l
  4. Roda a Dinâmica Inversa e a COMPARA com inverse_methods_moments dos autores
     (validação; responde ao ponto crítico 1 do Prof. Yusdel Díaz).
  5. Roda a Otimização Estática com atuadores de reserva.
  6. Roda o JRA do joelho direito com as forças musculares da SO.
  7. Extrai as variáveis no instante do pico de carga AP, SOMENTE na fase de apoio
     do pé direito (GRF vertical direita > 20 N), todas da mesma reconstrução.

CONVENÇÃO DE SINAL (confira e descreva na dissertação)
  knee_r_on_tibia_r_in_tibia_r_fx é a força que o fêmur exerce sobre a tíbia,
  expressa no referencial da tíbia (eixo x anterior). Uma reação POSTERIOR sobre a
  tíbia (fx negativo) corresponde à solicitação de cisalhamento ANTERIOR da tíbia,
  que é a restringida pelo LCA. O script grava:
    Cisalhamento_AP_Pico = maior valor de (-fx) na fase de apoio, em N (positivo).
  O extrator antigo usava idxmax(fx), ou seja, o sentido oposto.

COMO USAR
  1. Confira os caminhos em CONFIGURAÇÕES.
  2. Rode com MODO_TESTE = True: processa apenas P05 / trial0084 e imprime a
     comparação com os autores. Envie o relatório gerado.
  3. Se a validação estiver boa, mude para MODO_TESTE = False e rode tudo.
  4. Aponte o CAMINHO_EXCEL do fase2_validacao_final_Nkg.py para o novo Excel.

Requisitos: OpenSim 4.4 (opensim-cmd), pandas, numpy, scipy, openpyxl
"""

import glob
import os
import re
import subprocess
import sys
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# ============================================================================
# CONFIGURAÇÕES
# ============================================================================
OPENSIM_CMD = ''   # deixe vazio para localizar automaticamente


def localizar_opensim_cmd():
    """Procura o opensim-cmd.exe no PATH e nas pastas usuais de instalação."""
    import shutil
    if OPENSIM_CMD and os.path.isfile(OPENSIM_CMD):
        return OPENSIM_CMD
    no_path = shutil.which('opensim-cmd')
    if no_path:
        return no_path
    candidatos = []
    for raiz in ('C:\\', r'C:\Program Files', r'C:\Program Files (x86)',
                 os.path.expanduser('~')):
        candidatos += glob.glob(os.path.join(raiz, 'OpenSim*', 'bin', 'opensim-cmd.exe'))
    if not candidatos:
        raise FileNotFoundError(
            'opensim-cmd.exe não encontrado. No PowerShell, rode:\n'
            '  Get-ChildItem C:\\ -Filter opensim-cmd.exe -Recurse '
            '-ErrorAction SilentlyContinue | Select-Object FullName\n'
            'e cole o caminho em OPENSIM_CMD.')
    return sorted(candidatos)[-1]   # versão mais recente
PASTA_ZENODO = (r'C:\JG\opensim\Dados_Zenodo'
                r'\Nitschke_et_al_Change_of_Direction_Marker_Simulation')
PASTA_SAIDA = r'C:\JG\opensim\Resultados_FASE3'
PARTICIPANTES = [2]                   # amostra completa: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
TRIALS_SOMENTE = ['0122', '0123']     # vazio = todos os trials do v-cut de cada participante

MODO_TESTE = False                    # True: só o trial definido em TESTE
TESTE = ('05', '0084')

# Início antecipado da Otimização Estática e da Análise de Reação Articular (s).
# No P02 / trial0123, o equilíbrio inicial do músculo flex_dig_r não converge no
# primeiro quadro da janela (comprimento de fibra NaN). A SO e o JRA passam a
# começar antes do contato inicial, na fase de balanço do pé direito. A extração
# (fase4) continua restrita à janela dos autores. O 0122 entra como controle:
# comparar com a pasta P02_trial0122_original para confirmar que antecipar o
# início não altera os valores dentro da janela.
INICIO_ANTECIPADO_S = {('02', '0123'): 0.05, ('02', '0122'): 0.05}

FC_FILTRO = 15.0                      # Hz, como Nitschke et al.
LIMIAR_CONTATO = 20.0                 # N, GRF vertical para definir apoio
TIMEOUT_S = 3600

# Atuadores de reserva (força ótima; quanto menor, mais penalizado o uso)
RESERVA_PELVE = 10.0                  # resíduos da pelve
RESERVA_MEMBRO_INFERIOR = 1.0         # articulações com músculos
RESERVA_SEM_MUSCULO = 100.0           # tronco e braços (sem músculos no runMaD)
PREFIXOS_SEM_MUSCULO = ('lumbar', 'arm_', 'elbow', 'pro_sup', 'wrist')

# Fator aplicado à força isométrica máxima de TODOS os músculos, apenas na SO e
# no JRA (a ID não usa músculos). 1.0 = modelo original. No teste com 1.0, os
# vastos e o reto femoral saturaram (ativação 1,0) e a reserva do joelho chegou a
# 53 Nm (27% do momento). Critério de Hicks et al. (2015): reserva < 5% do pico
# do momento articular.
FATOR_FORCA_MUSCULAR = 2.0
LIMITE_RESERVA_PCT = 5.0


def modelo_com_forca_ajustada(caminho_modelo, fator, pasta):
    if fator == 1.0:
        return caminho_modelo
    arvore = ET.parse(caminho_modelo)
    for no in arvore.getroot().iter('max_isometric_force'):
        no.text = repr(float(no.text) * fator)
    nome = os.path.basename(caminho_modelo).replace('.osim', f'_forca_x{fator:g}.osim')
    destino = os.path.join(pasta, nome)
    arvore.write(destino, encoding='UTF-8', xml_declaration=True)
    return destino


# ============================================================================
# LEITURA E ESCRITA DE ARQUIVOS .mot / .sto
# ============================================================================
def ler_storage(caminho):
    em_graus = None
    with open(caminho, 'r', errors='ignore') as f:
        for i, linha in enumerate(f):
            if linha.lower().startswith('indegrees'):
                em_graus = 'yes' in linha.lower()
            if 'endheader' in linha.lower():
                break
    df = pd.read_csv(caminho, sep=r'\s+', skiprows=i + 1, engine='python')
    df.columns = [c.strip() for c in df.columns]
    return df, em_graus


def escrever_mot(df, caminho, em_graus=False):
    with open(caminho, 'w', newline='\n') as f:
        f.write(f'{os.path.basename(caminho)}\nversion=1\nnRows={len(df)}\n'
                f'nColumns={df.shape[1]}\ninDegrees={"yes" if em_graus else "no"}\n'
                'endheader\n')
        df.to_csv(f, sep='\t', index=False, float_format='%.8f')


def passa_baixa(sinal, fs, fc=FC_FILTRO, ordem=3):
    b, a = butter(ordem, fc / (fs / 2.0))
    return filtfilt(b, a, sinal)


# ============================================================================
# 1. GRF MEDIDAS: FILTRAGEM E CARGAS EXTERNAS COM MAPEAMENTO CORRETO
# ============================================================================
def preparar_grf(caminho_grf, pasta):
    grf, _ = ler_storage(caminho_grf)
    fs = 1.0 / np.median(np.diff(grf['time']))
    saida = grf.copy()
    for pre in ('', 'l_'):
        vy = grf[f'{pre}ground_force_vy'].to_numpy()
        contato_bruto = vy > LIMIAR_CONTATO
        for eixo in 'xyz':
            col = f'{pre}ground_force_v{eixo}'
            saida[col] = passa_baixa(grf[col].to_numpy(), fs)
            # Centro de pressão: mantém o último valor válido fora do contato
            cop = f'{pre}ground_force_p{eixo}'
            serie = grf[cop].where(contato_bruto).ffill().bfill().fillna(0.0)
            saida[cop] = passa_baixa(serie.to_numpy(), fs)
        sem_contato = saida[f'{pre}ground_force_vy'] < LIMIAR_CONTATO
        for eixo in 'xyz':
            saida.loc[sem_contato, f'{pre}ground_force_v{eixo}'] = 0.0

    caminho_mot = os.path.join(pasta, 'grf_medidas_filtradas.mot')
    escrever_mot(saida, caminho_mot)

    xml = f'''<?xml version="1.0" encoding="UTF-8" ?>
<OpenSimDocument Version="40000">
  <ExternalLoads name="externalloads">
    <objects>
      <ExternalForce name="GRF_direita">
        <applied_to_body>calcn_r</applied_to_body>
        <force_expressed_in_body>ground</force_expressed_in_body>
        <point_expressed_in_body>ground</point_expressed_in_body>
        <force_identifier>ground_force_v</force_identifier>
        <point_identifier>ground_force_p</point_identifier>
        <data_source_name>Unassigned</data_source_name>
      </ExternalForce>
      <ExternalForce name="GRF_esquerda">
        <applied_to_body>calcn_l</applied_to_body>
        <force_expressed_in_body>ground</force_expressed_in_body>
        <point_expressed_in_body>ground</point_expressed_in_body>
        <force_identifier>l_ground_force_v</force_identifier>
        <point_identifier>l_ground_force_p</point_identifier>
        <data_source_name>Unassigned</data_source_name>
      </ExternalForce>
    </objects>
    <groups />
    <datafile>{caminho_mot}</datafile>
  </ExternalLoads>
</OpenSimDocument>
'''
    caminho_xml = os.path.join(pasta, 'cargas_externas.xml')
    with open(caminho_xml, 'w', encoding='utf-8') as f:
        f.write(xml)
    return caminho_xml, saida


# ============================================================================
# 2. ATUADORES DE RESERVA
# ============================================================================
def coordenadas_livres(caminho_modelo, colunas_coord):
    raiz = ET.parse(caminho_modelo).getroot()
    livres = []
    for c in raiz.iter('Coordinate'):
        nome = c.get('name')
        travada = c.find('locked')
        if travada is not None and travada.text.strip().lower() == 'true':
            continue
        if nome in colunas_coord:
            livres.append(nome)
    return livres


def criar_reservas(coords, pasta):
    itens = []
    for c in coords:
        if c.startswith('pelvis'):
            f = RESERVA_PELVE
        elif c.startswith(PREFIXOS_SEM_MUSCULO):
            f = RESERVA_SEM_MUSCULO
        else:
            f = RESERVA_MEMBRO_INFERIOR
        itens.append(f'''      <CoordinateActuator name="reserva_{c}">
        <coordinate>{c}</coordinate>
        <optimal_force>{f}</optimal_force>
        <min_control>-10000</min_control>
        <max_control>10000</max_control>
      </CoordinateActuator>''')
    xml = ('<?xml version="1.0" encoding="UTF-8" ?>\n<OpenSimDocument Version="40000">\n'
           '  <ForceSet name="reservas">\n    <objects>\n' + '\n'.join(itens) +
           '\n    </objects>\n    <groups />\n  </ForceSet>\n</OpenSimDocument>\n')
    caminho = os.path.join(pasta, 'atuadores_reserva.xml')
    with open(caminho, 'w', encoding='utf-8') as f:
        f.write(xml)
    return caminho


# ============================================================================
# 3. SETUPS DO OPENSIM
# ============================================================================
def setup_id(modelo, coords, cargas, t0, t1, pasta):
    xml = f'''<?xml version="1.0" encoding="UTF-8" ?>
<OpenSimDocument Version="40000">
  <InverseDynamicsTool name="ID">
    <results_directory>{pasta}</results_directory>
    <model_file>{modelo}</model_file>
    <time_range>{t0} {t1}</time_range>
    <forces_to_exclude>Muscles</forces_to_exclude>
    <external_loads_file>{cargas}</external_loads_file>
    <coordinates_file>{coords}</coordinates_file>
    <lowpass_cutoff_frequency_for_coordinates>{FC_FILTRO}</lowpass_cutoff_frequency_for_coordinates>
    <output_gen_force_file>inverse_dynamics.sto</output_gen_force_file>
  </InverseDynamicsTool>
</OpenSimDocument>
'''
    caminho = os.path.join(pasta, 'setup_ID.xml')
    with open(caminho, 'w', encoding='utf-8') as f:
        f.write(xml)
    return caminho


def setup_analyze(nome, modelo, reservas, coords, cargas, t0, t1, pasta, analise):
    xml = f'''<?xml version="1.0" encoding="UTF-8" ?>
<OpenSimDocument Version="40000">
  <AnalyzeTool name="{nome}">
    <model_file>{modelo}</model_file>
    <replace_force_set>false</replace_force_set>
    <force_set_files>{reservas}</force_set_files>
    <results_directory>{pasta}</results_directory>
    <output_precision>8</output_precision>
    <initial_time>{t0}</initial_time>
    <final_time>{t1}</final_time>
    <solve_for_equilibrium_for_auxiliary_states>false</solve_for_equilibrium_for_auxiliary_states>
    <AnalysisSet name="Analyses">
      <objects>
{analise}
      </objects>
      <groups />
    </AnalysisSet>
    <ControllerSet name="Controllers">
      <objects />
      <groups />
    </ControllerSet>
    <external_loads_file>{cargas}</external_loads_file>
    <states_file />
    <coordinates_file>{coords}</coordinates_file>
    <speeds_file />
    <lowpass_cutoff_frequency_for_coordinates>{FC_FILTRO}</lowpass_cutoff_frequency_for_coordinates>
  </AnalyzeTool>
</OpenSimDocument>
'''
    caminho = os.path.join(pasta, f'setup_{nome}.xml')
    with open(caminho, 'w', encoding='utf-8') as f:
        f.write(xml)
    return caminho


def analise_so(t0, t1):
    return f'''        <StaticOptimization name="StaticOptimization">
          <on>true</on>
          <start_time>{t0}</start_time>
          <end_time>{t1}</end_time>
          <step_interval>1</step_interval>
          <in_degrees>true</in_degrees>
          <use_model_force_set>true</use_model_force_set>
          <activation_exponent>2</activation_exponent>
          <use_muscle_physiology>true</use_muscle_physiology>
          <optimizer_convergence_criterion>0.0001</optimizer_convergence_criterion>
          <optimizer_max_iterations>100</optimizer_max_iterations>
        </StaticOptimization>'''


def analise_jra(t0, t1, arquivo_forcas):
    return f'''        <JointReaction name="JointReaction">
          <on>true</on>
          <start_time>{t0}</start_time>
          <end_time>{t1}</end_time>
          <step_interval>1</step_interval>
          <in_degrees>true</in_degrees>
          <forces_file>{arquivo_forcas}</forces_file>
          <joint_names>knee_r</joint_names>
          <apply_on_bodies>child</apply_on_bodies>
          <express_in_frame>child</express_in_frame>
        </JointReaction>'''


def rodar(setup, rotulo, log):
    log(f'    {rotulo}...')
    r = subprocess.run([OPENSIM_CMD, 'run-tool', setup], capture_output=True,
                       text=True, timeout=TIMEOUT_S)
    with open(setup.replace('.xml', '_saida.log'), 'w', encoding='utf-8') as f:
        f.write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'{rotulo} falhou: {r.stderr.strip()[-800:]}')


# ============================================================================
# 4. VALIDAÇÃO DA DINÂMICA INVERSA CONTRA OS AUTORES
# ============================================================================
MOMENTOS_VALIDACAO = ['knee_angle_r_moment', 'hip_adduction_r_moment',
                      'hip_flexion_r_moment', 'ankle_angle_r_moment']


def validar_id(id_proprio, id_autores):
    linhas = []
    for c in MOMENTOS_VALIDACAO:
        if c not in id_proprio or c not in id_autores:
            continue
        ref = np.interp(id_proprio['time'], id_autores['time'], id_autores[c])
        dif = id_proprio[c].to_numpy() - ref
        linhas.append({
            'momento': c,
            'autores_min': ref.min(), 'autores_max': ref.max(),
            'proprio_min': id_proprio[c].min(), 'proprio_max': id_proprio[c].max(),
            'RMSE_Nm': float(np.sqrt(np.mean(dif ** 2))),
            'r': float(np.corrcoef(ref, id_proprio[c])[0, 1]),
        })
    return pd.DataFrame(linhas)


# ============================================================================
# 5. PROCESSAMENTO DE UM TRIAL
# ============================================================================
def localizar(pasta_part, padrao):
    achados = glob.glob(os.path.join(pasta_part, '**', padrao), recursive=True)
    if not achados:
        raise FileNotFoundError(f'Não encontrado: {padrao} em {pasta_part}')
    return achados[0]


def processar_trial(part, trial, log):
    pasta_part = os.path.join(PASTA_ZENODO, f'Participant_{part}')
    modelo = localizar(pasta_part, f'Participant_{part}.osim')
    coords = localizar(pasta_part, f'trial{trial}_inverse_methods_coordinates.mot')
    grf = localizar(pasta_part, f'trial{trial}_measured_GRFs.mot')
    id_ref = localizar(pasta_part, f'trial{trial}_inverse_methods_moments.sto')

    pasta = os.path.join(PASTA_SAIDA, f'P{part}_trial{trial}')
    os.makedirs(pasta, exist_ok=True)
    log(f'\n=== Participante {part} | trial {trial} ===')

    # Janela do movimento de interesse = janela da ID dos autores
    id_autores, _ = ler_storage(id_ref)
    t0, t1 = float(id_autores['time'].min()), float(id_autores['time'].max())
    log(f'    Janela: {t0:.4f} a {t1:.4f} s')

    cargas, grf_filtrada = preparar_grf(grf, pasta)
    df_coord, coord_em_graus = ler_storage(coords)
    livres = coordenadas_livres(modelo, set(df_coord.columns))
    reservas = criar_reservas(livres, pasta)

    # 5.1 Dinâmica Inversa + validação
    rodar(setup_id(modelo, coords, cargas, t0, t1, pasta), 'Dinâmica Inversa', log)
    id_proprio, _ = ler_storage(os.path.join(pasta, 'inverse_dynamics.sto'))
    val = validar_id(id_proprio, id_autores)
    val.insert(0, 'trial', trial)
    val.insert(0, 'participante', part)
    log(val.round(2).to_string(index=False))

    # 5.2 Otimização Estática
    modelo_so = modelo_com_forca_ajustada(modelo, FATOR_FORCA_MUSCULAR, pasta)
    log(f'    Força isométrica máxima x{FATOR_FORCA_MUSCULAR:g} na SO e no JRA')
    t0_sim = t0 - INICIO_ANTECIPADO_S.get((part, trial), 0.0)
    if t0_sim != t0:
        log(f'    SO e JRA iniciadas em {t0_sim:.4f} s ({1000 * (t0 - t0_sim):.0f} ms antes da janela)')
    rodar(setup_analyze('SO', modelo_so, reservas, coords, cargas, t0_sim, t1, pasta,
                        analise_so(t0_sim, t1)), 'Otimização Estática', log)
    forcas_so = os.path.join(pasta, 'SO_StaticOptimization_force.sto')
    so, _ = ler_storage(forcas_so)
    reservas_joelho = [c for c in so.columns if c.startswith('reserva_knee_angle_r')]
    reservas_quadril = [c for c in so.columns if c.startswith('reserva_hip_') and c.endswith('_r')]
    pico_res_joelho = float(so[reservas_joelho].abs().max().max()) if reservas_joelho else np.nan
    pico_res_quadril = float(so[reservas_quadril].abs().max().max()) if reservas_quadril else np.nan
    log(f'    Pico da reserva no joelho direito: {pico_res_joelho:.1f} Nm | '
        f'quadril direito: {pico_res_quadril:.1f} Nm')

    # 5.3 JRA com forças musculares da SO
    # Critério de Hicks et al. (2015) para as reservas, por articulação
    pct_res = {}
    for coord in ('knee_angle_r', 'hip_flexion_r', 'hip_adduction_r', 'ankle_angle_r'):
        col_res, col_id = f'reserva_{coord}', f'{coord}_moment'
        if col_res in so and col_id in id_proprio:
            pct_res[coord] = 100 * so[col_res].abs().max() / id_proprio[col_id].abs().max()
    log('    Reserva / pico do momento (%): ' +
        ' | '.join(f'{k} {v:.1f}%{" ACIMA DE 5%" if v > LIMITE_RESERVA_PCT else ""}'
                   for k, v in pct_res.items()))

    rodar(setup_analyze('JRA', modelo_so, reservas, coords, cargas, t0_sim, t1, pasta,
                        analise_jra(t0_sim, t1, forcas_so)), 'Joint Reaction', log)
    jra, _ = ler_storage(os.path.join(pasta, 'JRA_JointReaction_ReactionLoads.sto'))

    # 5.4 Extração restrita ao apoio do pé direito
    vy_dir = np.interp(jra['time'], grf_filtrada['time'], grf_filtrada['ground_force_vy'])
    apoio = vy_dir > LIMIAR_CONTATO
    if not apoio.any():
        raise RuntimeError('Nenhum quadro de apoio do pé direito dentro da janela.')
    base = 'knee_r_on_tibia_r_in_tibia_r_'
    carga_anterior = -jra[f'{base}fx']                  # positiva = solicitação anterior
    idx = carga_anterior[apoio].idxmax()
    t_pico = float(jra.loc[idx, 'time'])

    # Ângulos: mesmas coordenadas usadas na simulação, com o mesmo filtro
    fs_c = 1.0 / np.median(np.diff(df_coord['time']))
    def angulo(nome):
        v = passa_baixa(df_coord[nome].to_numpy(), fs_c)
        if coord_em_graus is False:
            v = np.degrees(v)
        return float(np.interp(t_pico, df_coord['time'], v))

    linha = {
        'Participante': part,
        'Movimento': trial,
        'Tempo_Pico_s': t_pico,
        'Inicio_Apoio_s': float(jra['time'][apoio].min()),
        'Fim_Apoio_s': float(jra['time'][apoio].max()),
        'Cisalhamento_AP_Pico': float(carga_anterior[idx]),            # N
        'Compressao_Y': float(-jra.loc[idx, f'{base}fy']),             # N, positiva = compressão
        'Cisalhamento_ML': float(jra.loc[idx, f'{base}fz']),           # N, sinal do referencial da tíbia
        'Momento_Aducao_Quadril': float(np.interp(t_pico, id_proprio['time'],
                                                  id_proprio['hip_adduction_r_moment'])),
        'Angulo_Flexao_Joelho': angulo('knee_angle_r'),
        'Angulo_Flexao_Quadril': angulo('hip_flexion_r'),
        'Angulo_Aducao_Quadril': angulo('hip_adduction_r'),
        'Angulo_Rotacao_Quadril': angulo('hip_rotation_r'),
        'Pico_Reserva_Joelho_Nm': pico_res_joelho,
        'Pico_Reserva_Quadril_Nm': pico_res_quadril,
        'Reserva_Joelho_pct': pct_res.get('knee_angle_r', np.nan),
    }

    # Desfechos candidatos (para a decisão com os orientadores), em dois recortes:
    # apoio inteiro e aceitação de peso (contato inicial até o pico de flexão do joelho)
    joelho = np.interp(jra['time'], df_coord['time'], df_coord['knee_angle_r'])
    t_apoio = jra['time'][apoio]
    i_flex = pd.Series(joelho, index=jra.index)[apoio].idxmin()   # flexão é negativa
    aceitacao = apoio & (jra['time'] <= jra.loc[i_flex, 'time'])
    for rotulo, mascara in (('Apoio', apoio), ('Aceitacao', aceitacao)):
        fx, mx = jra[f'{base}fx'][mascara], jra[f'{base}mx'][mascara]
        linha[f'{rotulo}_Reacao_AP_anterior_max_N'] = float(fx.max())
        linha[f'{rotulo}_Solicitacao_anterior_max_N'] = float((-fx).max())
        linha[f'{rotulo}_Momento_frontal_min_Nm'] = float(mx.min())
        linha[f'{rotulo}_Momento_frontal_max_Nm'] = float(mx.max())
    linha['Fim_Aceitacao_s'] = float(jra.loc[i_flex, 'time'])
    log('    Candidatos (aceitação de peso): '
        f'solicitação anterior máx {linha["Aceitacao_Solicitacao_anterior_max_N"]:.0f} N | '
        f'reação AP máx {linha["Aceitacao_Reacao_AP_anterior_max_N"]:.0f} N | '
        f'momento frontal {linha["Aceitacao_Momento_frontal_min_Nm"]:.1f} a '
        f'{linha["Aceitacao_Momento_frontal_max_Nm"]:.1f} Nm')
    log(f'    Pico AP (solicitação anterior): {linha["Cisalhamento_AP_Pico"]:.1f} N '
        f'em {t_pico:.3f} s | compressão {linha["Compressao_Y"]:.1f} N')
    return linha, val


# ============================================================================
# EXECUÇÃO
# ============================================================================
def listar_trials():
    lista = []
    for p in PARTICIPANTES:
        part = str(p).zfill(2)
        pasta_part = os.path.join(PASTA_ZENODO, f'Participant_{part}')
        arquivos = glob.glob(os.path.join(pasta_part, '**', 'vcut', '**',
                                          'trial*_inverse_methods_coordinates.mot'),
                             recursive=True)
        trials = sorted({re.search(r'trial(\d+)_', os.path.basename(a)).group(1)
                         for a in arquivos})
        if TRIALS_SOMENTE:
            trials = [t for t in trials if t in TRIALS_SOMENTE]
        lista += [(part, t) for t in trials]
    return lista


if __name__ == '__main__':
    os.makedirs(PASTA_SAIDA, exist_ok=True)
    relatorio = open(os.path.join(PASTA_SAIDA, 'fase3_relatorio.txt'), 'w', encoding='utf-8')

    def log(msg):
        print(msg)
        relatorio.write(msg + '\n')
        relatorio.flush()

    OPENSIM_CMD = localizar_opensim_cmd()
    log(f'OpenSim: {OPENSIM_CMD}')
    trials = [TESTE] if MODO_TESTE else listar_trials()
    log(f'Trials a processar: {len(trials)} {"(MODO TESTE)" if MODO_TESTE else ""}')

    linhas, validacoes, falhas = [], [], []
    for part, trial in trials:
        try:
            linha, val = processar_trial(part, trial, log)
            linhas.append(linha)
            validacoes.append(val)
        except Exception as e:
            falhas.append((part, trial, str(e)))
            log(f'    FALHA: {e}')

    if validacoes:
        pd.concat(validacoes).to_csv(os.path.join(PASTA_SAIDA, 'fase3_validacao_ID.csv'),
                                     index=False, sep=';', decimal=',')
    if linhas:
        df = pd.DataFrame(linhas)
        nome = 'dataset_lesoes_svm_TESTE.xlsx' if MODO_TESTE else 'dataset_lesoes_svm_v2.xlsx'
        df.to_excel(os.path.join(PASTA_SAIDA, nome), index=False)
        log(f'\nDataset salvo: {nome} ({len(df)} ciclos)')
        ap = df['Cisalhamento_AP_Pico']
        log(f'Cisalhamento AP (N): mínimo {ap.min():.0f} | mediana {ap.median():.0f} '
            f'| máximo {ap.max():.0f}')

    log(f'\nConcluídos: {len(linhas)} | Falhas: {len(falhas)}')
    for f in falhas:
        log(f'  P{f[0]} trial{f[1]}: {f[2]}')
    relatorio.close()

OpenSim: C:\OpenSim 4.5\bin\opensim-cmd.exe
Trials a processar: 2 

=== Participante 02 | trial 0122 ===
    Janela: 0.6229 a 1.5600 s
    Dinâmica Inversa...
participante trial                momento  autores_min  autores_max  proprio_min  proprio_max  RMSE_Nm    r
          02  0122    knee_angle_r_moment      -103.04       101.32       -42.69        99.71    10.35 0.96
          02  0122 hip_adduction_r_moment      -100.30        49.14       -99.37        16.92     7.05 0.98
          02  0122   hip_flexion_r_moment      -183.06        43.66       -77.09        42.45    19.39 0.87
          02  0122   ankle_angle_r_moment      -183.79         2.97      -183.95         2.46     1.14 1.00
    Força isométrica máxima x2 na SO e no JRA
    SO e JRA iniciadas em 0.5729 s (50 ms antes da janela)
    Otimização Estática...
    Pico da reserva no joelho direito: 0.0 Nm | quadril direito: 0.0 Nm
    Reserva / pico do momento (%): knee_angle_r 0.0% | hip_flexion_r 0.0% | hip_adduction_r 0.0% 

In [3]:
import numpy, pandas, scipy, sklearn, matplotlib, seaborn, sys
print(sys.version)
for m in (numpy, pandas, scipy, sklearn, matplotlib, seaborn):
    print(m.__name__, m.__version__)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
numpy 2.3.5
pandas 2.3.3
scipy 1.16.3
sklearn 1.7.2
matplotlib 3.10.6
seaborn 0.13.2
